# PPO from Scratch on CartPole-v1

This notebook runs the SimWorld Option A submission: PPO implemented from scratch with PyTorch and Gymnasium. It uses the repository code in `src/ppo_cartpole.py`, trains on `CartPole-v1`, saves logs/curves, and evaluates the final policy over 100 episodes.

## 1. Setup

If this notebook is opened directly in Colab without the rest of the repository, fill in `REPO_URL` below and run the clone cell. If you opened it from a cloned repository, leave `REPO_URL` empty and continue.

In [ ]:
REPO_URL = 'https://github.com/0ozcharles/simworld-ppo-cartpole.git'
REPO_DIR = 'simworld-ppo-cartpole'

from pathlib import Path

if REPO_URL:
    if Path('src/ppo_cartpole.py').exists():
        !git pull --ff-only
    elif Path(REPO_DIR).exists():
        %cd {REPO_DIR}
        !git pull --ff-only
    else:
        !git clone {REPO_URL} {REPO_DIR}
        %cd {REPO_DIR}

assert Path('src/ppo_cartpole.py').exists(), 'Run this notebook from the repository root or set REPO_URL above.'

In [ ]:
!pip -q install -r requirements.txt

## 2. Train PPO

The default settings are tuned for `CartPole-v1` and should run on free Colab. GPU is optional for this environment.

In [ ]:
import sys
sys.path.append('.')

from src.ppo_cartpole import PPOConfig, train

config = PPOConfig(
    env_id='CartPole-v1',
    seed=42,
    total_timesteps=500_000,
    num_envs=8,
    num_steps=256,
    update_epochs=8,
    learning_rate=1.0e-3,
    ent_coef=0.001,
    eval_episodes=100,
    output_dir='runs',
    device='auto',
)

summary = train(config)
summary

## 3. Inspect Training Curve

In [ ]:
from IPython.display import Image, display
import json
import matplotlib.pyplot as plt

training_curve_path = Path(summary['run_dir']) / 'reward_curve.png'
evaluation_curve_path = Path(summary['run_dir']) / 'evaluation_curve.png'

if not evaluation_curve_path.exists():
    evaluation_json = Path(summary['run_dir']) / 'evaluation.json'
    evaluation = json.loads(evaluation_json.read_text())
    episode_returns = evaluation.get('episode_returns', [])
    if episode_returns:
        mean_return = sum(episode_returns) / len(episode_returns)
        plt.figure(figsize=(9, 5))
        plt.plot(episode_returns, marker='o', linewidth=1.5, label='Evaluation return')
        plt.axhline(mean_return, color='tab:green', linestyle='-', linewidth=2, label=f'Mean {mean_return:.1f}')
        plt.axhline(450, color='tab:red', linestyle='--', linewidth=1, label='Target 450')
        plt.xlabel('Evaluation episode')
        plt.ylabel('Return')
        plt.title('Deterministic Policy Evaluation on CartPole-v1')
        plt.legend()
        plt.tight_layout()
        plt.savefig(evaluation_curve_path, dpi=160)
        plt.close()

print('Training rollout curve:')
display(Image(filename=str(training_curve_path)))

print('Final deterministic evaluation curve:')
if evaluation_curve_path.exists():
    display(Image(filename=str(evaluation_curve_path)))
else:
    print('evaluation_curve.png was not generated, but final metrics are shown below.')

## 4. Show Final Metrics

In [ ]:
import json

print(json.dumps(summary, indent=2))
print('\nFiles generated in:', summary['run_dir'])